## 简单用法

#### 读取预料

In [1]:
import jieba
from glove_simple import Glove, Corpus

课程代码使用的是 glove_python 库（ from glove import Glove, Corpus ），但它基于 Cython，最后一次更新是 2018 年，与 Python 3.12 完全无法兼容。

我创建了一个 纯 Python 实现的替代模块 glove_simple.py ，提供了完全相同的 API：

- Corpus — 构建共现矩阵（支持 dictionary 、 window 、 ignore_missing 参数）
- Glove — 训练词向量（支持 no_components 、 learning_rate 、 alpha 、 max_count 、 max_loss 等参数）
- 支持 save() / load() 保存加载模型

#### 1.内存方式

In [2]:
# 加载自定义词典
jieba.load_userdict('data/phone_dict.txt')

# 停用词
filepath = 'data/stopwords.txt'
stopwords = [line.strip() for line in open(filepath, 'r', encoding='utf-8')]

Building prefix dict from the default dictionary ...
Loading model from cache C:\Users\Gkk\AppData\Local\Temp\jieba.cache
Loading model cost 0.451 seconds.
Prefix dict has been built successfully.


In [3]:
# 读取文件
def rf2wl(filepath):
    cut_list = []
    with open(filepath, 'r', encoding='utf-8') as f:
        for line in f.readlines():
            line = line.strip()
            seg_list = jieba.cut(line)
            seg_list = [word for word in seg_list if word not in stopwords and word != ' ']
            cut_list.append(seg_list)
    return cut_list

In [4]:
# 未分词语料
filepath = 'data/mb.txt'
sentences = rf2wl(filepath)

### 计算共现矩阵

```python
corpus_model = Corpus(dictionary=None)
corpus_model.fit(corpus, window=10, ignore_missing=False)
```
* dictionary：词典到 id 的映射输入，格式为{word1:index1,word2:index2,word3:index3...}
* corpus：训练语料，格式为[[word1,word2,word3...],[word1,word2,word3...],...]，列表中每一个子列表为分完词的一篇文档
* window：窗口大小
* ignore_missing：针对 OOV 词的处理，在指定上述 dictionary 的情况下，如果出现未登录词，此值为False时会报KeyError错误，如果值为True，则会保留结果，不会报错


In [5]:
corpus_model = Corpus(dictionary=None)

In [6]:
corpus_model.fit(sentences,window=5)

In [7]:
# 获取词袋
print(corpus_model.dictionary)

OrderedDict({'Apple': 0, 'iPhone': 1, 'Plus': 2, 'A1864': 3, '64GB': 4, '深空灰': 5, '色': 6, '移动': 7, '联通': 8, '电信': 9, '4G': 10, '手机': 11, 'A1661': 12, '128G': 13, '黑色': 14, 'OPPO': 15, 'KTx': 16, '双模': 17, '5G': 18, '4800万': 19, '四摄': 20, '5000mAh': 21, '长': 22, '续航': 23, '90Hz': 24, '电竞屏': 25, '蓝影': 26, '6GB': 27, '128GB': 28, '30W': 29, '闪充': 30, '全网通': 31, '游戏': 32, '智能手机': 33, '一加': 34, 'OnePlus': 35, '8T': 36, '旗舰': 37, '120Hz': 38, '柔性': 39, '直屏': 40, '65W': 41, '高通': 42, '骁龙865': 43, '超强': 44, '12GB': 45, '256GB': 46, '青域': 47, '拍照': 48, '游戏手机': 49, '小米': 50, '红米': 51, '全面屏': 52, '拍照手机': 53, '版': 54, '3GB': 55, '32GB': 56, '金色': 57, '双卡双待': 58, 'A1660': 59, 'X': 60, 'A1865': 61, '红米Note5A': 62, '4GB': 63, '铂银灰': 64, '荣耀': 65, 'V10': 66, '标配': 67, '幻夜黑': 68, '全': 69, '画屏': 70, 'Redmi': 71, 'Note': 72, '天玑800U': 73, '18W': 74, '快充': 75, '超清三摄': 76, '云墨灰': 77, '6s': 78, 'A1699': 79, '玫瑰金': 80, 'Note3': 81, '美颜双摄': 82, '12': 83, 'A2404': 84, '支持': 85, '10': 86, 'GT': 87, '加速': 88, 'A

### 训练glove词向量

```python
glove = Glove(no_components=30, learning_rate=0.05,
        alpha=0.75, max_count=100, max_loss=10.0,
        random_state=None)

glove.fit(matrix, epochs=5, no_threads=2, verbose=False)

In [8]:
glove = Glove(no_components=200,learning_rate=0.05)
glove.fit(corpus_model.matrix, epochs=10, no_threads=1, verbose=True)
glove.add_dictionary(corpus_model.dictionary)

Epoch 1/10, loss = 0.051708
Epoch 2/10, loss = 0.035162
Epoch 3/10, loss = 0.032403
Epoch 4/10, loss = 0.031428
Epoch 5/10, loss = 0.030902
Epoch 6/10, loss = 0.030497
Epoch 7/10, loss = 0.030201
Epoch 8/10, loss = 0.029949
Epoch 9/10, loss = 0.029708
Epoch 10/10, loss = 0.029484


### 保存模型

In [9]:
glove.save('models/glove_model.pkl')
corpus_model.save('models/corpus_model.pkl')

### 加载模型


In [ ]:
model = Glove.load('models/glove_model.pkl')
corpus_model = Corpus.load('models/corpus_model.pkl')



In [13]:
# 获取词向量矩阵
print(model.word_vectors)

[[-1.58277600e-03  2.93220468e-03 -1.46777807e-03 ... -2.48119350e-04
  -1.17990061e-04 -2.51202833e-03]
 [ 1.76168154e-03  4.92682763e-03 -1.53458174e-03 ... -4.68312406e-03
  -1.25495543e-03 -2.36291027e-03]
 [-9.69651922e-04  2.74556532e-03 -9.83361011e-04 ... -4.58757332e-04
   2.04651155e-03 -3.37902277e-04]
 ...
 [ 1.78149861e-03 -2.05690690e-03 -1.48194750e-03 ...  2.20761756e-03
  -4.78119066e-04  9.16264943e-04]
 [ 5.83958006e-04 -1.14071811e-03  5.15397541e-04 ... -2.97629697e-03
  -2.10344205e-03 -2.90251303e-03]
 [-1.51649758e-03 -1.29106691e-03 -1.17573878e-03 ...  2.03087875e-03
   5.27576423e-05  5.21492592e-05]]


In [16]:
# 获取单词word2vec值
model.word_vectors[glove.dictionary['Apple']]

array([-1.58277600e-03,  2.93220468e-03, -1.46777807e-03, -1.06943482e-03,
       -1.87374470e-03, -4.22140480e-03,  7.59234663e-04, -1.49995820e-03,
        3.46872870e-03,  7.56563097e-04,  4.91843260e-04,  3.48117889e-03,
       -1.90238356e-03,  4.35186935e-03, -1.55553399e-04, -3.44303455e-03,
        3.31152880e-04,  2.41726546e-03, -1.17201809e-03,  1.00915937e-03,
        5.55552558e-04, -7.41320956e-04,  1.39642377e-03,  2.61665235e-03,
       -1.88665303e-03, -1.91265892e-04, -9.10560747e-04, -4.96887376e-06,
       -4.15632963e-04, -4.40139268e-03,  1.53524858e-03,  1.70938138e-03,
        9.32455373e-05,  1.38594544e-03, -3.15935235e-03,  2.39882697e-05,
       -9.62078612e-05,  2.13981185e-03, -3.57989030e-03, -4.07463382e-03,
       -1.66589996e-03,  1.02035296e-04,  2.26639714e-03,  1.57892240e-03,
       -2.23030468e-03, -3.64758254e-04,  2.53215936e-03, -3.92767151e-03,
        9.51560545e-04,  2.05802635e-03,  3.03089349e-03,  2.31465282e-03,
       -1.73907016e-04, -